# RL Masterclass 01: Foundations — MDPs & the Bellman Equation

**Track:** Reinforcement Learning (0 to Masterclass)  
**Prerequisites:** Python, basic probability  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"In supervised learning, you have a dataset of correct answers. In RL, the agent must DISCOVER the right answers through trial and error. But how does an agent know if a move in chess was good when the game doesn't end for another 40 moves?"*

---

## Why Learn RL as an AI Engineer?

You might think RL is only for games and robots. In reality:

| Application | RL Technique | Where in This Curriculum |
|-------------|-------------|-------------------------|
| **ChatGPT/Claude alignment** | RLHF, PPO, DPO | NB12, RL/06 |
| **Recommendation ranking** | Contextual bandits | NB35 System Design |
| **Ad bidding** | Multi-armed bandits | Production ML |
| **Chip design** (Google TPU) | AlphaChip | Research |
| **Robotics** | PPO, SAC | Sim-to-real |

**The RLHF connection is the main reason you need RL.** Without understanding PPO (RL/05), the alignment notebook (NB12) is a black box.

## Table of Contents
1. The RL Problem: Agent, Environment, Reward
2. Markov Decision Processes (MDPs)
3. The Bellman Equation — The Foundation of All RL
4. Dynamic Programming: Value Iteration & Policy Iteration

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors see RL as "AI plays games." Seniors see RL as the mathematical framework for SEQUENTIAL decision making under uncertainty. This framework underlies everything from LLM alignment (how to teach GPT to be helpful) to real-time bidding (how to maximize ad revenue).

**Mental Model:** Imagine teaching a dog a new trick. You can't EXPLAIN the trick to the dog (no supervised learning). Instead, you give treats (rewards) when it does something right and withhold treats when it doesn't. Over thousands of repetitions, the dog learns. RL is the math behind this process.

**Common Junior Pitfall:** Confusing RL with supervised learning. In SL, you tell the model the correct answer. In RL, the model must DISCOVER the correct answer through trial and error. This makes RL much harder to train and much more sample-inefficient.

---

In [ ]:
!pip install -q numpy matplotlib

## 1 · The RL Framework

Every RL problem has three components:

```
                    action (aₜ)
    ┌─────────┐  ──────────────>  ┌─────────────┐
    │  AGENT  │                   │ ENVIRONMENT │
    │  (brain)│  <──────────────  │  (world)    │
    └─────────┘   state (sₜ₊₁)   └─────────────┘
                  reward (rₜ₊₁)
```

The agent loop:
1. **Observe** the current state $s_t$
2. **Choose** an action $a_t$ based on its policy $\pi(a|s)$
3. **Receive** a reward $r_{t+1}$ and new state $s_{t+1}$
4. **Learn** from this experience to improve the policy
5. **Repeat** until the episode ends

### Terminology

| Term | Meaning | Chess Example |
|------|---------|---------------|
| **State** ($s$) | Current situation | Board position |
| **Action** ($a$) | What the agent does | Move a piece |
| **Reward** ($r$) | Immediate feedback | +1 win, -1 loss, 0 ongoing |
| **Policy** ($\pi$) | Strategy: state → action | "If opponent threatens queen, defend" |
| **Episode** | One complete interaction | One full game |
| **Return** ($G$) | Total cumulative reward | Did we win? |
| **Discount** ($\gamma$) | How much future rewards matter | 0.99 = plan ahead, 0.1 = greedy |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =============================================
# A Simple Environment: GridWorld
# =============================================
# The agent starts at (0,0) and must reach the goal at (3,3)
# Each step gives -1 reward (incentivizing shortest path)
# Reaching the goal gives +10

class GridWorld:
    """A 4x4 grid environment.
    
    This is the simplest possible RL environment — perfect for
    understanding MDPs before moving to complex problems.
    """
    
    def __init__(self, size=4):
        self.size = size
        self.goal = (size-1, size-1)  # Bottom-right corner
        self.reset()
    
    def reset(self):
        """Start a new episode at top-left corner."""
        self.pos = (0, 0)
        return self.pos
    
    def step(self, action):
        """Take an action: 0=up, 1=right, 2=down, 3=left.
        Returns: (new_state, reward, done)
        """
        moves = {0: (-1,0), 1: (0,1), 2: (1,0), 3: (0,-1)}
        dx, dy = moves[action]
        new_x = max(0, min(self.size-1, self.pos[0] + dx))
        new_y = max(0, min(self.size-1, self.pos[1] + dy))
        self.pos = (new_x, new_y)
        
        if self.pos == self.goal:
            return self.pos, +10, True   # Reached goal!
        return self.pos, -1, False       # Each step costs -1

# Demo: Random agent
env = GridWorld(size=4)
state = env.reset()
total_reward = 0
actions_taken = []
action_names = ['↑', '→', '↓', '←']

print('Random agent playing GridWorld:')
for step in range(20):
    action = np.random.randint(4)  # Random policy!
    next_state, reward, done = env.step(action)
    total_reward += reward
    actions_taken.append(action_names[action])
    if done:
        print(f'  Reached goal in {step+1} steps! Total reward: {total_reward}')
        break
else:
    print(f'  Failed to reach goal in 20 steps. Reward: {total_reward}')

print(f'  Path: {" ".join(actions_taken)}')
print(f'\n  A random policy is terrible. We need LEARNING.')
print(f'  Optimal path: → → → ↓ ↓ ↓ (6 steps, reward = -6 + 10 = 4)')

---
## 2 · Markov Decision Process (MDP)

An MDP formalizes the RL problem as a tuple: $(S, A, P, R, \gamma)$

| Component | Symbol | Meaning |
|-----------|--------|--------|
| States | $S$ | All possible situations |
| Actions | $A$ | All possible moves |
| Transition | $P(s'|s,a)$ | Probability of reaching $s'$ from $s$ via $a$ |
| Reward | $R(s,a,s')$ | Immediate payoff |
| Discount | $\gamma \in [0,1]$ | How much to value future rewards |

### The Markov Property

The key assumption: **the future depends only on the present, not the past.**

$$P(s_{t+1} | s_t, a_t, s_{t-1}, a_{t-1}, ...) = P(s_{t+1} | s_t, a_t)$$

In plain English: if you know the current board position in chess, you don't need to know HOW you got there. The current state contains all relevant information.

### Why the Discount Factor $\gamma$ Matters

In [ ]:
# Demonstrating the discount factor
import numpy as np
import matplotlib.pyplot as plt

# The RETURN is the total discounted reward from time t:
# G_t = r_{t+1} + γ * r_{t+2} + γ² * r_{t+3} + ...

def compute_return(rewards, gamma):
    """Compute discounted return."""
    G = 0
    for r in reversed(rewards):
        G = r + gamma * G
    return G

# Scenario: robot gets reward +1 at every step for 100 steps
rewards = [1.0] * 100
gammas = [0.0, 0.5, 0.9, 0.99, 1.0]

print('Discount factor effect on total return:')
print(f'{"γ":>8} {"Return":>10} {"Behavior":>30}')
print('-' * 50)
for g in gammas:
    G = compute_return(rewards, g)
    behavior = {
        0.0: 'Only cares about immediate',
        0.5: 'Short-term focused',
        0.9: 'Balanced (common default)',
        0.99: 'Long-term planner',
        1.0: 'Infinite horizon (risky)',
    }[g]
    print(f'{g:>8.2f} {G:>10.1f} {behavior:>30}')

print(f'\nγ=0.99 is the most common choice for complex tasks.')
print(f'γ=1.0 can cause infinite returns in non-terminating environments.')

# Visualize discount decay
fig, ax = plt.subplots(figsize=(10, 4))
steps = np.arange(50)
for g in [0.5, 0.9, 0.95, 0.99]:
    ax.plot(steps, g**steps, label=f'γ={g}', lw=2)
ax.set_xlabel('Steps into the future')
ax.set_ylabel('Discount weight')
ax.set_title('How Much Does the Agent Care About Future Rewards?')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'At γ=0.9, a reward 20 steps away is worth only 0.9²⁰ = {0.9**20:.3f} of its face value.')

---
## 3 · The Bellman Equation

### The Foundation of ALL Reinforcement Learning

The **value** of a state $s$ under policy $\pi$ is the expected return starting from $s$:

$$V^\pi(s) = \mathbb{E}_\pi \left[ \sum_{t=0}^{\infty} \gamma^t r_{t+1} \mid s_0 = s \right]$$

The Bellman equation breaks this into two parts — **immediate reward + discounted future value**:

$$\boxed{V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V^\pi(s') \right]}$$

In plain English: **The value of where I am = what I get now + γ × the value of where I'll be.**

This is recursive! The value of a state depends on the values of the states it leads to. This recursion is what makes RL work — and what ALL RL algorithms (Q-learning, PPO, etc.) are ultimately solving.

### 🎓 ANSWERING THE DEEP QUESTION

**Q:** *How does an agent know if a chess move was good when the game doesn't end for 40 more moves?*

**A:** Through the Bellman equation! The agent doesn't wait until the game ends. It estimates the VALUE of each state ("how likely am I to win from this position?"). A good move transitions to a high-value state; a bad move transitions to a low-value state. These value estimates propagate BACKWARD from the final reward through the Bellman recursion.

---
## 4 · Dynamic Programming: Solving Small MDPs Exactly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def value_iteration(grid_size=4, gamma=0.9, threshold=1e-6):
    """Value Iteration: find optimal value function for GridWorld.
    
    This is the simplest way to solve an MDP when you know
    the transition probabilities (model-based RL).
    
    Algorithm:
        1. Initialize V(s) = 0 for all states
        2. For each state, update:
           V(s) = max_a Σ P(s'|s,a) [R(s,a,s') + γ V(s')]
        3. Repeat until V converges (changes < threshold)
    """
    n = grid_size
    V = np.zeros((n, n))
    goal = (n-1, n-1)
    moves = {0: (-1,0), 1: (0,1), 2: (1,0), 3: (0,-1)}
    
    iterations = 0
    while True:
        V_new = np.zeros_like(V)
        for i in range(n):
            for j in range(n):
                if (i, j) == goal:
                    V_new[i, j] = 0  # Terminal state
                    continue
                
                # Try all actions, keep the best
                action_values = []
                for a, (di, dj) in moves.items():
                    ni, nj = max(0,min(n-1,i+di)), max(0,min(n-1,j+dj))
                    reward = 10 if (ni,nj) == goal else -1
                    action_values.append(reward + gamma * V[ni, nj])
                
                V_new[i, j] = max(action_values)  # Bellman optimality!
        
        # Check convergence
        delta = np.max(np.abs(V_new - V))
        V = V_new
        iterations += 1
        if delta < threshold:
            break
    
    return V, iterations

# Solve GridWorld
V_star, iters = value_iteration(grid_size=4, gamma=0.9)

print(f'Value Iteration converged in {iters} iterations')
print(f'\nOptimal Value Function V*(s):')
print(np.round(V_star, 1))
print(f'\nReading the grid:')
print(f'  - State (0,0) has value {V_star[0,0]:.1f}: expected total reward from start')
print(f'  - State (3,3) has value 0: it\'s the goal (episode ends)')
print(f'  - Values increase toward the goal: the agent should follow the gradient')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Value function heatmap
im = axes[0].imshow(V_star, cmap='RdYlGn')
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{V_star[i,j]:.1f}', ha='center', va='center', fontsize=12, fontweight='bold')
axes[0].set_title('Optimal Value Function V*(s)', fontweight='bold')
plt.colorbar(im, ax=axes[0])

# Optimal policy (arrows)
moves = {0: (-1,0), 1: (0,1), 2: (1,0), 3: (0,-1)}
arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
axes[1].imshow(np.ones((4,4)), cmap='Greys', alpha=0.1)
for i in range(4):
    for j in range(4):
        if (i,j) == (3,3):
            axes[1].text(j, i, '★', ha='center', va='center', fontsize=24)
            continue
        best_a = max(range(4), key=lambda a: V_star[max(0,min(3,i+moves[a][0])), max(0,min(3,j+moves[a][1]))])
        axes[1].text(j, i, arrows[best_a], ha='center', va='center', fontsize=24)
axes[1].set_title('Optimal Policy π*(s)', fontweight='bold')

plt.tight_layout()
plt.savefig('rl_value_iteration.png', dpi=100)
plt.show()
print(f'Every arrow points toward the goal — the agent learned the shortest path!')

---
## Limitation of Dynamic Programming

Value Iteration works perfectly for GridWorld (16 states). But:

| Game | # States | DP Feasible? |
|------|----------|-------------|
| GridWorld (4×4) | 16 | ✅ |
| Tic-Tac-Toe | ~5,000 | ✅ |
| Chess | ~10⁴⁷ | ❌ |
| Go | ~10¹⁷⁰ | ❌ |
| Real-world robotics | ∞ (continuous) | ❌ |

For large/continuous state spaces, we need **model-free** methods that learn from EXPERIENCE rather than computing values for every state. That's Q-learning (RL/02) and deep RL (RL/03-05).

---

## ✅ Knowledge Check

### Q1: Markov Property
In a card game, does the current hand of cards satisfy the Markov property? Why or why not?

<details><summary>👉 Answer</summary>

It depends on what you define as the state. If the state is ONLY the cards in your hand, then NO — which cards have already been played matters (the past affects the future). If the state includes the cards in your hand AND all previously played cards, then YES — the full game state contains all information needed for future decisions. In practice, you expand the state definition until Markov holds.
</details>

### Q2: Discount factor
An agent in a stock trading environment uses γ=0.3. What kind of trading strategy does this produce?

<details><summary>👉 Answer</summary>

A very short-term strategy (day trading). γ=0.3 means the agent barely values rewards more than 2-3 steps into the future (0.3³ = 0.027 ≈ 0). The agent will optimize for immediate gains, ignoring long-term portfolio growth. For long-term investing, you'd want γ=0.99+.
</details>

### Q3: Bellman equation
In the Bellman equation V(s) = max_a [R + γ V(s')], what happens to V(s) when γ = 0?

<details><summary>👉 Answer</summary>

V(s) = max_a R(s,a). The agent only considers immediate rewards and completely ignores the value of future states. This is a purely greedy strategy — it takes the action with the highest immediate payoff, with no planning ahead.
</details>

---

## 🔨 Practical Practice

### Exercise 1: Policy Iteration
Implement Policy Iteration (the other DP algorithm): 1) Start with a random policy, 2) Compute V for that policy, 3) Improve the policy greedily, 4) Repeat until stable. Compare convergence speed to Value Iteration.

### Exercise 2: Stochastic Transitions
Modify GridWorld so actions are stochastic: 80% chance of moving in the intended direction, 10% left, 10% right. How does the optimal policy change?

### Exercise 3: Reward Shaping
Add a dangerous cell at (1,1) with reward -10. Observe how the optimal policy routes AROUND the danger. Then try reward -100. Does the path change?

---

**Next →** RL 02: Q-Learning & Temporal Difference Methods